# Merge & split datasets

Single pipeline that:
1. **separates** the 2025 questionnaire into its sub-samples (physics, high school, psychology, teacher, altro dipartimento),
2. **merges** them with the earlier cohorts into the cross-year *all-psychology* files and the *human* / *GPT* databases,
3. processes the **2019** experts / high-school data,
4. **splits** the relevant samples into low/high math-anxiety groups by the MAS-IT median.

The cue-words summary table lives in a separate notebook (`cue_words_summary.ipynb`).

In [1]:
import re
import numpy as np
import pandas as pd

from config import RESSERENO_ROOT, MATH_ANXIETY_ROOT
from questionnaires_analysis import low_high_anxiety_division

FORMATTING = RESSERENO_ROOT / "out" / "formatting"
MA_FORMATTING = MATH_ANXIETY_ROOT / "out" / "formatting"

## Load the source data

In [2]:
# 2025 questionnaire (every 2025 sub-sample lives in these two files)
mas_it_2025 = pd.read_csv(FORMATTING / "all_students_2025" / "mas_it_2025.csv")
assoc_2025 = pd.read_csv(
    FORMATTING / "all_students_2025" / "associations_valence_task_2025.csv"
)

# earlier psychology cohorts (merged into the cross-year "all psychology" sample)
mas_it_2023 = pd.read_csv(MA_FORMATTING / "humans_2023" / "mas_it_2023_humans.csv")
mas_it_2024 = pd.read_csv(MA_FORMATTING / "humans_2024" / "mas_it_2024_humans.csv")
mas_it_BA_2024 = pd.read_csv(FORMATTING / "psychology_BA_2024" / "mas_it_2024.csv")

assoc_2023 = pd.read_csv(
    MA_FORMATTING / "humans_2023" / "associations_valence_2023_humans.csv"
)
assoc_2024 = pd.read_csv(
    MA_FORMATTING / "humans_2024" / "associations_valence_2024_humans.csv"
)
assoc_BA_2024 = pd.read_csv(
    FORMATTING / "psychology_BA_2024" / "associations_valence_2024.csv"
)

## Separate the 2025 sub-samples

Each 2025 sub-sample is identified by its `participant_id` prefix. `fisica` covers
`fisica` + `fisica_unitn` + `fisica_stem`; `liceo` covers `liceo` + `liceo_sci` + `liceo_lin`.

In [3]:
# 2025 sub-sample -> participant_id prefix
SAMPLES_2025 = {
    "physics": "fisica",
    "high_school": "liceo",
    "psychology": "interfacce",  # feeds the cross-year all-psychology merge
    "teacher": "teacher",
    "altro_dipartimento": "altro",
}


def by_prefix(df, prefix):
    return df[df["participant_id"].astype(str).str.startswith(prefix)].reset_index(
        drop=True
    )


mas_it = {name: by_prefix(mas_it_2025, pref) for name, pref in SAMPLES_2025.items()}
assoc = {name: by_prefix(assoc_2025, pref) for name, pref in SAMPLES_2025.items()}

for name in SAMPLES_2025:
    print(f"{name:20s}: {len(mas_it[name]):3d} participants")

physics             :  24 participants
high_school         :  62 participants
psychology          :  23 participants
teacher             :  27 participants
altro_dipartimento  :   3 participants


In [4]:
# save the per-sample 2025 files (paths kept identical to the previous pipeline; teacher /
# altro dipartimento are new folders). psychology is only used in the merge below, so not saved here.
SAMPLE_OUTPUTS = {
    "physics": (
        "physics_2025",
        "mas_it_fisica_2025.csv",
        "associations_valence_fisica.csv",
    ),
    "high_school": (
        "liceo_2025",
        "mas_it_liceo_2025.csv",
        "associations_valence_liceo_2025.csv",
    ),
    "teacher": (
        "teacher_2025",
        "mas_it_teacher_2025.csv",
        "associations_valence_teacher_2025.csv",
    ),
    "altro_dipartimento": (
        "altro_dipartimento_2025",
        "mas_it_altro_2025.csv",
        "associations_valence_altro_2025.csv",
    ),
}

for name, (folder, mas_name, assoc_name) in SAMPLE_OUTPUTS.items():
    out_dir = FORMATTING / folder
    out_dir.mkdir(parents=True, exist_ok=True)
    mas_it[name].to_csv(out_dir / mas_name, index=False)
    assoc[name].to_csv(out_dir / assoc_name, index=False)

## Cross-year *all psychology* sample

In [4]:
# 2023 + 2024 + BA 2024 + 2025 psychology, aligned to the 2025 columns
cols_2025 = mas_it["psychology"].columns
psy_mas_cohorts = [mas_it_2024, mas_it_2023, mas_it_BA_2024, mas_it["psychology"]]
psy_assoc_cohorts = [assoc_2024, assoc_2023, assoc_BA_2024, assoc["psychology"]]

all_psychology_mas_it = pd.concat(
    [d[cols_2025.intersection(d.columns)] for d in psy_mas_cohorts], ignore_index=True
)
all_psychology_assoc = pd.concat(psy_assoc_cohorts, ignore_index=True)

psy_dir = FORMATTING / "all_psychology_students"
all_psychology_mas_it.to_csv(psy_dir / "mas_it_all_psy_students.csv", index=False)
all_psychology_assoc.to_csv(
    psy_dir / "associations_valence_all_psy_students.csv", index=False
)
print("all psychology students:", len(all_psychology_mas_it))

all psychology students: 316


## Human & GPT databases

Each sample's `participant_id`s are renumbered to `{sample}_001, {sample}_002, …` and the sub-samples
are concatenated into the merged `database/` files.

In [5]:
def renumber(mas_it_df, assoc_df, sample):
    """Renumber a sample's ids as '{sample}_NNN' and propagate the mapping to its associations."""
    mas_new = mas_it_df.reset_index(drop=True).copy()
    new_ids = [f"{sample}_{i:03d}" for i in range(1, len(mas_new) + 1)]
    id_map = dict(zip(mas_new["participant_id"], new_ids))
    mas_new["participant_id"] = new_ids

    assoc_new = assoc_df.copy()
    assoc_new["participant_id"] = assoc_df["participant_id"].map(id_map)
    return mas_new, assoc_new


def build_database(samples, out_prefix):
    """Renumber every sample and write the merged mas-it / associations database files."""
    mas_parts, assoc_parts = zip(
        *(
            renumber(mas_df, assoc_df, name)
            for name, (mas_df, assoc_df) in samples.items()
        )
    )
    all_mas = pd.concat(mas_parts, ignore_index=True)
    all_assoc = pd.concat(assoc_parts, ignore_index=True)

    db_dir = FORMATTING / "database"
    all_mas.to_csv(db_dir / f"{out_prefix}_mas_it.csv", index=False)
    all_assoc.to_csv(db_dir / f"{out_prefix}_associations_valence.csv", index=False)
    return all_mas, all_assoc

In [6]:
# human database: cross-year psychology + the 2025 samples
HUMAN_DB = {
    "psychology": (all_psychology_mas_it, all_psychology_assoc),
    "physics": (mas_it["physics"], assoc["physics"]),
    "high_school": (mas_it["high_school"], assoc["high_school"]),
    "teacher": (mas_it["teacher"], assoc["teacher"]),
    "altro_dipartimento": (mas_it["altro_dipartimento"], assoc["altro_dipartimento"]),
}
all_humans_mas_it, all_humans_assoc = build_database(HUMAN_DB, "all_humans")
print("all humans:", len(all_humans_mas_it))

all humans: 432


In [7]:
# GPT-oss database
gpt_dir = FORMATTING / "gpt_oss"
GPT_SOURCES = {
    "gpt_oss_psychology": (
        "mas_it_psychology.csv",
        "associations_valence_psychology.csv",
    ),
    "gpt_oss_physics": ("mas_it_physics.csv", "associations_valence_physics.csv"),
    "gpt_oss_high_school": ("mas_it_liceo.csv", "associations_valence_liceo.csv"),
}
GPT_DB = {
    name: (pd.read_csv(gpt_dir / mas_name), pd.read_csv(gpt_dir / assoc_name))
    for name, (mas_name, assoc_name) in GPT_SOURCES.items()
}
all_gpt_mas_it, all_gpt_assoc = build_database(GPT_DB, "all_gpt")
print("all gpt:", len(all_gpt_mas_it))

all gpt: 402


## 2019 experts & high-school data

Parse the raw 2019 text files into the long `associations_valence_{sample}.csv` format used elsewhere.

In [9]:
DATA_2019 = RESSERENO_ROOT / "data" / "data_2019"


def cue_word_extraction(file):
    """Read a 2019 associations file into (cue_word, association) pairs + the cue-word list."""
    pairs = [re.findall(r'"([^"]+)"', line) for line in open(file).readlines()]
    df = pd.DataFrame(pairs, columns=["association_0", "association_1"])
    counts = df["association_0"].value_counts()
    cue_words = counts[: len(counts) // 2].index.tolist()
    return df, cue_words


def _assign_valence(row, col, valence_map):
    word = row[col]
    if word in valence_map and valence_map[word]:
        return float(valence_map[word].pop(0).replace(".", ""))
    return np.nan


def association_valence_df(valence_file, df, sample):
    """Attach the 2019 valence ratings to a pairs dataframe and reshape to long format."""
    rows = [
        v.strip().split("\t")
        for v in open(DATA_2019 / f"{valence_file}.txt").readlines()
    ]
    valence_map = {}
    for row in rows:
        if not row[0].replace(".", "").isdigit():
            valence_map.setdefault(row[0], []).extend(
                [v for v in row[1:] if v.replace(".", "").isdigit()]
            )

    df = df.copy()
    df["valence_association_1"] = df.apply(
        _assign_valence, axis=1, args=("association_1", valence_map)
    )
    df["valence_association_0"] = df.apply(
        _assign_valence, axis=1, args=("association_0", valence_map)
    )
    df["new_index"] = df.index + 1
    df["cue_word_original"] = df["association_0"]

    df_long = pd.wide_to_long(
        df,
        stubnames=["association", "valence_association"],
        i=["new_index", "cue_word_original"],
        j="index",
        sep="_",
        suffix=r"\d+",
    ).reset_index()
    df_long = df_long.rename(
        columns={
            "association": "word",
            "valence_association": "valence",
            "cue_word_original": "cue_word",
        }
    )
    for col in ["cue_word", "word"]:
        df_long[col] = (
            df_long[col]
            .astype(str)
            .str.lower()
            .str.replace(r"[\d\W_]+", " ", regex=True)
            .str.strip()
        )
    df_long = df_long.dropna()
    df_long = df_long[(df_long["word"] != "nan") & (df_long["word"] != "")]

    out_dir = FORMATTING / sample
    out_dir.mkdir(parents=True, exist_ok=True)
    df_long.to_csv(out_dir / f"associations_valence_{sample}.csv", index=False)
    return df_long

In [10]:
students_df, _ = cue_word_extraction(DATA_2019 / "Students_unFiltered.txt")
experts_df, _ = cue_word_extraction(DATA_2019 / "Experts_unfiltered.txt")

association_valence_df("ValenceStudentsUPD", students_df, "students_2019")
association_valence_df("ValenceExperts", experts_df, "experts_2019")

,new_index,cue_word,index,word,valence
0,1,island,0,island,3.0
1,1,island,1,sea,3.0
3,2,land rover,1,island,3.0
5,3,land rover,1,sea,5.0
6,4,island,0,island,3.0
...,...,...,...,...,...
6361,3181,herb,1,trimmer,3.0
6363,3182,herb,1,vegetables,5.0
6366,3184,collapse,0,collapse,3.0
6367,3184,collapse,1,destroy,2.0


## Split by MAS-IT median (low / high anxiety)

`low_high_anxiety_division` writes `high_anx.csv` / `low_anx.csv` into `out/splitted_dfs/{sample}/`.
Add or remove a line to change which samples are split.

In [8]:
# sample name (= splitted_dfs subfolder) -> (mas-it df, associations df)
SPLIT_SAMPLES = {
    "all_psychology_students": (all_psychology_mas_it, all_psychology_assoc),
    "high_school_2025": (mas_it["high_school"], assoc["high_school"]),
    "psychology_gpt_oss": GPT_DB["gpt_oss_psychology"],
    "high_school_gpt_oss": GPT_DB["gpt_oss_high_school"],
}

for sample, (mas_df, assoc_df) in SPLIT_SAMPLES.items():
    low_high_anxiety_division(mas_df, assoc_df, sample, RESSERENO_ROOT)


Results for all_psychology_students sample 

total median of the summed scores: 37.0
number of total students: 316
number of students with high anxiety: 150
number of students with low anxiety: 151
number of students to exclude: 15

Results for high_school_2025 sample 

total median of the summed scores: 31.0
number of total students: 62
number of students with high anxiety: 30
number of students with low anxiety: 27
number of students to exclude: 5

Results for psychology_gpt_oss sample 

total median of the summed scores: 80.0
number of total students: 316
number of students with high anxiety: 143
number of students with low anxiety: 157
number of students to exclude: 16

Results for high_school_gpt_oss sample 

total median of the summed scores: 81.0
number of total students: 62
number of students with high anxiety: 28
number of students with low anxiety: 29
number of students to exclude: 5


## Participants summary

In [12]:
with open(FORMATTING / "participants_rassereno.txt", "a") as f:
    f.write(
        "Number of participants for Rassereno article:\n"
        "Experts 2019: 59\n"
        "High school students 2019: 159\n"
        f"High school students 2025: {len(mas_it['high_school'])}\n"
        f"Physics students 2025: {len(mas_it['physics'])}\n"
        f"Teachers 2025: {len(mas_it['teacher'])}\n"
        f"All psychology students: {len(all_psychology_mas_it)} "
        f"(2025 psy students: {len(mas_it['psychology'])})\n"
    )